# Тема 9 — Docker для LLM inference: контракт, который можно перевезти

## Образ — это чертёж передвижной кухни

**Что случилось в Topic 08.** Вы собрали шлюз (gateway) поверх замороженного профиля: валидация,
ограниченная очередь с честным 429, таймауты, отмена. Но жил сервис в «works on my machine»
земле: pip-установки на лету, какой Python — такой хост и дал, cwd решает, что увидит приложение.

**Проблема:** сервис, поведение которого зависит от хоста, не имеет контракта. «Вчера работало» —
не воспроизводимость; плавающий тег `latest` — отложенная поломка зависимостей; базовый тег
образа — не версия.

**Что меняется сейчас.** Приложение становится неизменяемым образом (image) с pinned
зависимостями: образ = упорядоченные слои (layer), порядок слоёв = политика build cache,
multi-stage держит компиляторы вне рантайма, non-root и лимиты — дефолты безопасности.
Метафора темы — «больше кухонь»: образ это чертёж передвижной кухни, контейнер — работающая
кухня, слои — её модули. Рецепт собирается снизу вверх, и перестановка модулей сбрасывает весь
приготовленный путь.

**Ваша задача на лабу.** (1) Заморозить план сборки ДО первой сборки: порядок COPY-слоёв и
предсказания инвалидации кэша. (2) Поднять наивный v0-образ и получить /health через границу
контейнера — первый работающий артефакт. (3) Собрать lean v1: multi-stage, `.dockerignore`,
non-root, healthcheck. (4) Прогнать весь T08-контракт через границу: health, completion, 429.
(5) Измерить холодный и тёплый старт и сократить доминирующий член одним обоснованным
изменением. (6) Разобрать две поломки по журналу: tag drift и OOM (exit 137). Stretch: SBOM
и риски цепочки поставок.

**Правила игры.** (1) Все docker-клетки охраняются: без локального демона они печатают причину
пропуска и ничего не делают — валидация остаётся зелёной. (2) Размеры и секунды — ВАШИ измерения
вашего образа на вашей машине; курс не цитирует их как engine-бенчмарки. (3) GPU-вариант
(CUDA base, vLLM pin, `--gpus`) — карточки конфигурации и демонстрация владельца, не
студенческий шаг. (4) Предсказание замораживается до измерения — как в Topics 06–07.

Рантайм: **локальный Docker**, CPU-образы для всех. Конверт: 2 контактных часа + 3–5 часов
самостоятельной работы; на диске нужно ~2 GiB под образы.

## ⚙ Как работать с лабой: локальный Docker (прочитайте до первой docker-клетки)

Рантайм этой лабы — ВАШ локальный Docker: Docker Desktop на macOS/Windows или Docker Engine
на Linux. CPU-образы: мок-движок внутри контейнера не требует GPU. Jupyter — локальный, не
Colab: клетки пишут файлы в `build/` (контекст сборки) и в `results/` (журнал) рядом с ноутбуком.

**Проверка готовности — одна команда в терминале:**

```bash
docker info
```

Ответ пришёл — всё готово. Требования: `docker` на PATH, демон запущен, ~2 GiB свободного диска.

**Что будет, если демона нет.** Каждая docker-клетка охраняется проверкой `docker info`
с коротким таймаутом (свёрнутый хелпер `require_docker()`). Без демона клетка печатает
`skipped: локальный Docker недоступен — выполните этот шаг на своей машине (см. гид в начале)`
и больше ничего не делает. Поэтому валидация ноутбука (nbclient без Docker) остаётся зелёной,
а план-режим — предсказания, staged-файлы, журналы, графики, разбор стенограмм — исполняется
везде.

**GPU — не в этой лабе.** Продакшн-вариант образа (CUDA base + vLLM pin + `--gpus`) существует
в курсе как карточки конфигурации и демонстрация владельца с сохранёнными доказательствами.
Студенческий шаг — только CPU-мок: он собирается и отвечает на любой машине, где есть Docker.

In [ ]:
import json, os, subprocess, time, hashlib, urllib.error, urllib.request
from pathlib import Path
from tqdm.auto import tqdm  # правило курса: видимый прогресс в каждой клетке дольше ~5 с
from IPython.display import display

PROFILE = 'local'  # лаба исполняется на локальном Docker; без демона — план-режим
ATTEMPT = 'attempt-001'
RESULTS = os.path.join('results', PROFILE, ATTEMPT)
BUILD = 'build'    # контекст сборки: сюда попадают app/, Dockerfile, .dockerignore
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(os.path.join(BUILD, 'app'), exist_ok=True)
print(f'profile={PROFILE}  журнал={RESULTS}  контекст сборки={BUILD}/')

In [ ]:
DOCKER_SKIP = ('skipped: локальный Docker недоступен — '
               'выполните этот шаг на своей машине (см. гид в начале)')

def docker(*args: str) -> tuple[int, str]:
    """Run one docker CLI command and return (returncode, combined output).

    Parameters
    ----------
    *args : str
        Arguments after the verb, e.g. docker('build', '-f', 'build/Dockerfile', 'build').

    Returns
    -------
    tuple of (int, str)
        Process return code and the combined stdout+stderr tail (last 4000 chars).
    """
    proc = subprocess.run(['docker', *args], capture_output=True, text=True, timeout=1800)
    return proc.returncode, ((proc.stdout + proc.stderr)[-4000:])

def require_docker(timeout_s: float = 10.0) -> bool:
    """Probe the local docker daemon with a short-timeout `docker info`.

    Parameters
    ----------
    timeout_s : float, default 10.0
        Wall budget for the probe; a missing binary also means False.

    Returns
    -------
    bool
        True only if the daemon answered successfully.
    """
    try:
        proc = subprocess.run(['docker', 'info'], capture_output=True, text=True, timeout=timeout_s)
    except (FileNotFoundError, subprocess.TimeoutExpired, OSError):
        return False
    return proc.returncode == 0

def wait_healthy(url: str, deadline_s: float = 120.0) -> float:
    """Poll one HTTP endpoint until it answers 200, with a visible progress bar.

    Parameters
    ----------
    url : str
        Health endpoint, e.g. 'http://127.0.0.1:8099/health'.
    deadline_s : float, default 120.0
        Wall budget before giving up.

    Returns
    -------
    float
        Waited seconds until the first 200 answer.

    Raises
    ------
    AssertionError
        If the endpoint never answered within the budget.
    """
    started = time.monotonic()
    with tqdm(total=deadline_s, desc='ждём /health', unit=' с') as bar:
        while True:
            try:
                with urllib.request.urlopen(url, timeout=3) as resp:
                    if resp.status == 200:
                        return round(time.monotonic() - started, 1)
            except (urllib.error.URLError, OSError):
                pass
            assert time.monotonic() - started < deadline_s, f'{url} не ответил за {deadline_s} с'
            time.sleep(1.0)
            bar.update(1.0)

## Блок 0 — Образ = упорядоченные слои (механизм до формул)

Каждая инструкция Dockerfile создаёт слой (layer): дельту над предыдущим слоем. Образ — стопка
слоёв в порядке записи. Кухня собирается снизу вверх: фундамент (base image), коммуникации
(системные библиотеки), оборудование (зависимости pip) и только потом ваши инструменты (код).

**Механизм build cache** — главная идея блока: cache hit для слоя возможен только если ни он сам,
ни один слой ВЫШЕ него не изменились. Отсюда таблица порядка:

| Правка | Что пересоберётся | Почему |
|---|---|---|
| файл кода в app/ | только верхний слой | всё ниже не менялось |
| app/requirements.txt | слой pip и выше | COPY зависимостей видит новое содержимое |
| тег базового образа | вся стопка | фундамент сменился |

**Вопрос для предсказания ДО сборки.** Какой из трёх порядков инструкций даёт cache hit слоя
зависимостей при каждой правке кода?

- A: `COPY . .` → `RUN pip install -r app/requirements.txt`
- B: `COPY app/requirements.txt …` → `RUN pip install …` → `COPY app/ app/`
- C: `RUN pip install …` → `COPY . .`

Заморозьте выбор и предсказания в build-plan.json (клетка ниже) ДО первой сборки — как прогноз
колена в Topic 06. Формулы блока (размер образа как сумма слоёв, холодный старт как сумма
членов) появятся в книге ПОСЛЕ того, как вы проверите предсказания живым `docker history`:
сначала механизм, потом сумма.

In [ ]:
APP_MAIN = """'T09 lab gateway: FastAPI plus MockEngine with declared synthetic latency, CPU-only.'
import asyncio
import os

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

# Declared latency knobs (env): T09_LATENCY_MS base cost, T09_QUEUE_MAX admission ceiling.
# Declared latency model: total_seconds = (LATENCY_MS + 10 * max_tokens) / 1000.
LATENCY_MS = float(os.environ.get('T09_LATENCY_MS', '200'))
QUEUE_MAX = int(os.environ.get('T09_QUEUE_MAX', '4'))

class CompletionRequest(BaseModel):
    'One completion request: prompt text and an output-token cap.'
    prompt: str
    max_tokens: int = 16

app = FastAPI(title='t09-gateway')
_in_flight = 0

@app.get('/health')
async def health() -> dict:
    'Liveness plus queue occupancy; the image HEALTHCHECK targets this route.'
    return {'status': 'ok', 'in_flight': _in_flight, 'queue_max': QUEUE_MAX}

@app.post('/v1/completions')
async def completions(req: CompletionRequest) -> dict:
    'One mock completion; 429 with Retry-After when the bounded queue is full.'
    global _in_flight
    if _in_flight >= QUEUE_MAX:
        raise HTTPException(status_code=429, detail='queue full',
                            headers={'Retry-After': '1'})
    _in_flight += 1
    try:
        await asyncio.sleep((LATENCY_MS + 10.0 * req.max_tokens) / 1000.0)
        return {'text': 'MOCK-COMPLETION',
                'usage': {'prompt_tokens': len(req.prompt.split()),
                          'completion_tokens': req.max_tokens}}
    finally:
        _in_flight -= 1
"""
APP_REQUIREMENTS = """fastapi==0.115.8
uvicorn==0.34.0
"""

def stage_app() -> dict:
    """Write the shipped gateway app into the build context (build/app/).

    The app is PROVIDED, not a student task: FastAPI + MockEngine with declared
    latency knobs. The Docker decisions around it are the lab's exercises.

    Returns
    -------
    dict
        Staged relative paths mapped to their sizes in bytes.
    """
    staged = {}
    for rel, text in (('app/main.py', APP_MAIN), ('app/requirements.txt', APP_REQUIREMENTS)):
        path = Path(BUILD) / rel
        path.write_text(text, encoding='utf-8')
        staged[rel] = path.stat().st_size
    return staged

# pandas DataFrame: живые строки контракта приложения ДО любой математики (принцип 9).
import pandas as pd
_contract = [
    dict(route='/health', method='GET', answer='200 {"status": "ok"}',
         promise='демон опрашивает healthcheck'),
    dict(route='/v1/completions', method='POST', answer='200 mock completion',
         promise='основной маршрут контракта T08'),
    dict(route='/v1/completions', method='POST, очередь полна', answer='429 + Retry-After',
         promise='честная перегрузка вместо вечной очереди'),
]
_staged = stage_app()
display(pd.DataFrame(_contract).style.set_caption(
    'контракт, который должен пережить границу контейнера; staged: '
    + ', '.join(f'{k} ({v} B)' for k, v in _staged.items())
    + '; ручки мока: T09_LATENCY_MS, T09_QUEUE_MAX'))

In [ ]:
def plan_cache_order() -> dict:
    """Freeze the build plan BEFORE the first build: layer order and cache predictions.

    Returns
    -------
    dict
        chosen_order ('A'/'B'/'C'), per-edit invalidation predictions and the
        cold/warm startup terms — all frozen into build-plan.json by the caller.
    """
    options = {
        'A': ['COPY . .', 'RUN pip install -r app/requirements.txt'],
        'B': ['COPY app/requirements.txt app/requirements.txt',
              'RUN pip install -r app/requirements.txt',
              'COPY app/ app/'],
        'C': ['RUN pip install -r app/requirements.txt', 'COPY . .'],
    }
    # Уровень A: почти написано. Выберите порядок слоёв — впишите одну букву.
    chosen_order = 'TODO'  # 'A', 'B' или 'C' из options выше
    return dict(
        chosen_order=chosen_order,
        options=options,
        predictions=dict(
            edit_app_code='TODO: какие слои пересоберутся и почему?',
            edit_requirements='TODO: какие слои пересоберутся и почему?',
            change_base_tag='TODO: какие слои пересоберутся и почему?'),
        startup_terms=dict(
            cold='TODO: члены холодного старта через + (например: pull + extract + ...)',
            warm='TODO: члены тёплого старта через +'),
        why='TODO: одно предложение — почему выбранный порядок бережёт cache pip-слоя')

_plan = plan_cache_order()
with open(os.path.join(RESULTS, 'build-plan.json'), 'w', encoding='utf-8') as fh:
    json.dump(_plan, fh, ensure_ascii=False, indent=1)
print(f"build-plan.json заморожен ДО первой сборки: порядок слоёв = {_plan['chosen_order']}")

## Блок 1 — Сборка: сначала наивный v0, потом ваш lean v1

**Почему v0 существует.** Первый работающий артефакт важнее первого красивого. v0 — сознательно
наивный Dockerfile (полный base, `COPY . .` до pip, root, без healthcheck): он собирается,
отвечает на /health — и становится базовой линией всех измерений. Это клетка-доказательство,
что цикл «образ → контейнер → ответ через границу» у вас работает.

**Четыре дефекта v0, которые чинит ваш v1:**

1. полный base (`python:3.12`, ~1 GB на диске) вместо slim — каждый мегабайт платится на каждом pull;
2. `COPY . .` ДО pip-слоя — любая правка кода сбрасывает cache зависимостей (разрушьте план блока 0);
3. root внутри контейнера — лишняя поверхность атаки;
4. нет HEALTHCHECK — демон не знает, жив ли сервис.

**v1 — multi-stage:** builder-стадия ставит зависимости в `--prefix=/install`; runtime-стадия
копирует только `/install` и код. Компиляторы и pip-кэш остаются в builder — в образ они не
попадают. Плюс `.dockerignore` (контекст без журналов и venv), non-root user, HEALTHCHECK
на /health. Каждый слой v1 вы защитите одной фразой — «зачем он в образе».

Оценка времени: первая сборка v0 ~1–3 мин (тянется полный base), повторные — секунды.

In [ ]:
NAIVE_DOCKERFILE = """# v0: works on my machine — deliberately naive starting point (provided infra)
FROM python:3.12
WORKDIR /srv
COPY . .
RUN pip install --no-cache-dir -r app/requirements.txt
ENV T09_LATENCY_MS=200 T09_QUEUE_MAX=4
EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""
if not require_docker():
    print(DOCKER_SKIP)
else:
    (Path(BUILD) / 'Dockerfile.v0').write_text(NAIVE_DOCKERFILE, encoding='utf-8')
    started = time.monotonic()
    # Оценка времени: ~1–3 мин на первой сборке (полный base ~1 GB), секунды на повторных.
    for _step in tqdm((1,), desc='docker build v0', unit=' этап'):
        rc, out = docker('build', '-f', 'build/Dockerfile.v0', '-t', 't09-gw:v0', BUILD)
    print(f'build v0: rc={rc} за {time.monotonic() - started:.0f} c'
          if rc == 0 else f'build v0 ОШИБКА rc={rc}: {out[-300:]}')

In [ ]:
if not require_docker():
    print(DOCKER_SKIP)
else:
    docker('rm', '-f', 't09-v0')
    docker('run', '-d', '--name', 't09-v0', '-p', '8098:8000', 't09-gw:v0')
    waited = wait_healthy('http://127.0.0.1:8098/health', deadline_s=120.0)
    with urllib.request.urlopen('http://127.0.0.1:8098/health', timeout=5) as resp:
        health = json.loads(resp.read())
    docker('rm', '-f', 't09-v0')
    print(f"первый работающий артефакт: /health ответил за {waited} c "
          f"(in_flight={health['in_flight']}, queue_max={health['queue_max']}) — "
          'контракт отвечает через границу контейнера')

## Справочник: термины и команды блока

| Термин | Что это |
|---|---|
| image | неизменяемый артефакт: упорядоченные слои + манифест |
| layer | дельта файловой системы от одной инструкции Dockerfile |
| build cache | переиспользование слоя, пока вся стопка выше не менялась |
| multi-stage build | компиляторы в builder-стадии; в рантайм копируются только артефакты |
| .dockerignore | список того, что не попадает в контекст сборки |
| entrypoint / CMD | команда, которую запускает контейнер |
| healthcheck | декларация живости, которую опрашивает демон |
| non-root | процесс внутри контейнера под непривилегированным пользователем |
| pull / extract | скачать слои из registry / распаковать их на диск |
| OOM, exit 137 | cgroup-лимит памяти: ядро убило процесс сигналом SIGKILL |
| SBOM | software bill of materials: инвентарь пакетов образа |

**Docker CLI, который понадобится:** `docker build -f … -t … CONTEXT`, `docker run -d --name …
-p host:container IMAGE`, `docker ps`, `docker images`, `docker history IMAGE`, `docker logs NAME`,
`docker stop/start/rm NAME`, `docker inspect NAME`. Из ноутбука их вызывает свёрнутый хелпер
`docker(...)` — не набирайте руками то, что должно попасть в журнал.

In [ ]:
def write_dockerfile_stages() -> str:
    """Build the v1 Dockerfile: lean multi-stage, pinned deps, non-root, healthcheck.

    Returns
    -------
    str
        Full Dockerfile text for the v1 image; the caller writes it to build/Dockerfile.
    """
    # Уровень B: скелет готов — заполните шаги по комментариям.
    # Шаг 1: builder-стадия: FROM python:3.12-slim AS deps; WORKDIR /srv
    # Шаг 2: COPY app/requirements.txt первым, затем
    #        RUN pip install --no-cache-dir --prefix=/install -r app/requirements.txt
    # Шаг 3: runtime-стадия: FROM python:3.12-slim; WORKDIR /srv; useradd --create-home appuser
    # Шаг 4: COPY --from=deps /install /usr/local; COPY app/ app/; USER appuser
    # Шаг 5: EXPOSE 8000; HEALTHCHECK на http://127.0.0.1:8000/health;
    #        CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
    # Соберите строки в один текст (join с переводом строки) и верните его.
    raise NotImplementedError('TODO: соберите multi-stage Dockerfile по шагам')

_text = write_dockerfile_stages()
(Path(BUILD) / 'Dockerfile').write_text(_text, encoding='utf-8')
print(f'Dockerfile v1 записан: {len(_text.splitlines())} строк — multi-stage, non-root, healthcheck')

In [ ]:
DOCKERIGNORE = [
    '# журналы и виртуальные окружения не должны попадать в контекст сборки',
    'results/',
    '__pycache__/',
    'TODO',  # впишите: каталог виртуальных окружений Python (шаблон на venv)
    'TODO',  # впишите: служебный каталог git
]

_missing = [i for i, p in enumerate(DOCKERIGNORE) if p == 'TODO']
if _missing:
    print(f'заполните .dockerignore: позиции {_missing} со значением TODO — файл не записан')
else:
    (Path(BUILD) / '.dockerignore').write_text('\n'.join(DOCKERIGNORE) + '\n', encoding='utf-8')
    print(f'.dockerignore записан: {len(DOCKERIGNORE)} строк — контекст без журналов и окружений')

In [ ]:
if not require_docker():
    print(DOCKER_SKIP)
else:
    started = time.monotonic()
    # Оценка времени: ~10–60 с (slim-base обычно уже локален после v0? нет — другой тег,
    # первая сборка v1 может тянуть slim-образ; повторные — секунды).
    for _step in tqdm((1,), desc='docker build v1', unit=' этап'):
        rc, out = docker('build', '-f', 'build/Dockerfile', '-t', 't09-gw:v1', BUILD)
    print(f'build v1: rc={rc} за {time.monotonic() - started:.0f} c — образ t09-gw:v1 готов'
          if rc == 0 else f'build v1 ОШИБКА rc={rc}: {out[-300:]}')

## Блок 2 — Граница контейнера: тот же контракт, другой мир

Контейнер живёт в своих сетевых пространствах. Чтобы достучаться из ноутбука: порт контейнера
отображают на порт хоста (`-p host:container`), переменные конфигурации передают через `-e`
(ручки мока `T09_LATENCY_MS`, `T09_QUEUE_MAX`), лимит памяти — через `--memory` (admission
control для контейнера).

**Проверка контракта.** В Topic 08 шлюз обещал: /health → 200, /v1/completions → 200,
перегрузка → 429 + Retry-After. Теперь тот же набор зондов уходит на localhost:mapped-port.
Если зонд проходит БЕЗ изменений — контракт пережил границу; в этом и есть смысл образа:
поведение больше не зависит от хоста.

**Лимиты — это admission control.** `--memory 512m` обещает контейнеру стену: превысил — решает
ядро (OOM, exit 137). Эта механика понадобится в блоке поломок; само слово «лимит» здесь значит
то же, что и в Topic 08 у очереди: честный отказ лучше бесконечного ожидания.

In [ ]:
RUN_FLAGS = dict(
    name='t09-gw',
    port_host='TODO',     # впишите свободный порт хоста, например 8099
    port_container=8000,
    memory_limit='TODO',  # впишите лимит admission control, например 512m
    latency_env='TODO',   # впишите env-ручку мока, например T09_LATENCY_MS=200
)

def run_command(flags: dict) -> list:
    """Assemble the `docker run` argument list from the chosen flags.

    Parameters
    ----------
    flags : dict
        RUN_FLAGS after filling: name, port_host, port_container, memory_limit, latency_env.

    Returns
    -------
    list of str
        Arguments after `docker`, e.g. ['run', '-d', '--name', ..., '-p', ..., IMAGE].
    """
    # Уровень B: скелет готов — соберите аргументы по шагам.
    # Шаг 1: ['run', '-d', '--name', flags['name']]
    # Шаг 2: добавьте '-p', f"{flags['port_host']}:{flags['port_container']}"
    # Шаг 3: добавьте '--memory', flags['memory_limit'] и '-e', flags['latency_env']
    # Шаг 4: завершите именем образа 't09-gw:v1'
    raise NotImplementedError('TODO: соберите команду запуска по шагам')

if not require_docker():
    print(DOCKER_SKIP)
else:
    _todo = [k for k, v in RUN_FLAGS.items() if str(v).startswith('TODO')]
    if _todo:
        print(f'заполните RUN_FLAGS: {_todo} — docker run не выполнялся')
    else:
        cmd = run_command(RUN_FLAGS)
        docker('rm', '-f', RUN_FLAGS['name'])
        rc, out = docker(*cmd)
        waited = wait_healthy(f"http://127.0.0.1:{RUN_FLAGS['port_host']}/health", deadline_s=90.0)
        print(f"контейнер {RUN_FLAGS['name']} запущен (rc={rc}), "
              f'/health через границу за {waited} c')

In [ ]:
def probe_container(base_url: str, concurrency: int = 16, timeout_s: float = 30.0) -> list:
    """Run the T08 probe suite against the mapped port: health, completion, 429 overload.

    Parameters
    ----------
    base_url : str
        Gateway root, e.g. 'http://127.0.0.1:8099'.
    concurrency : int, default 16
        Simultaneous completion requests for the overload probe (QUEUE_MAX=4 → часть 429).
    timeout_s : float, default 30.0
        Wall budget for one request.

    Returns
    -------
    list of dict
        One row per probe: probe, expected, http_status, ok.
    """
    # Уровень B: скелет готов.
    # Шаг 1: GET base_url + '/health' → ждём 200.
    # Шаг 2: POST base_url + '/v1/completions' с {"prompt": "ping", "max_tokens": 8} → ждём 200.
    # Шаг 3: ThreadPoolExecutor(concurrency) тех же POST → часть получает 429 (очередь QUEUE_MAX=4).
    # Шаг 4: верните строки dict(probe=..., expected=..., http_status=..., ok=...).
    raise NotImplementedError('TODO: реализуйте три зонда контракта')

if not require_docker():
    print(DOCKER_SKIP)
else:
    _rows = probe_container(f"http://127.0.0.1:{RUN_FLAGS['port_host']}")
    with open(os.path.join(RESULTS, 'probe-journal.json'), 'w', encoding='utf-8') as fh:
        json.dump(_rows, fh, ensure_ascii=False, indent=1)
    display(pd.DataFrame(_rows).style.set_caption(
        'контракт T08 через границу контейнера (probe-journal.json)'))

In [ ]:
def summarize_layers(history_text: str) -> list:
    """Itemize docker history output into per-layer rows for size accounting.

    Parameters
    ----------
    history_text : str
        Lines of `docker history --format SIZE,TAB,CreatedBy`, newest layer first.

    Returns
    -------
    list of dict
        Rows: size_mb (float, 0.0 for metadata layers), instruction (short).
    """
    # Уровень B: скелет готов.
    # Шаг 1: разберите text на строки; каждую строку — по табуляции на размер и инструкцию.
    # Шаг 2: размер приведите к MB: '129.8MB' → 129.8, '452KB' → 0.5, '0B' → 0.0.
    # Шаг 3: инструкцию обрежьте до ~70 символов; пустую замените на '(metadata layer)'.
    # Шаг 4: верните список dict(size_mb=..., instruction=...).
    raise NotImplementedError('TODO: разберите вывод docker history по шагам')

if not require_docker():
    print(DOCKER_SKIP)
else:
    _img_rc, images_out = docker('images', '--format', '{{.Repository}}:{{.Tag}} {{.Size}}')
    _hist_rc, history_out = docker('history', '--format', '{{.Size}}\t{{.CreatedBy}}', 't09-gw:v1')
    _layer_rows = summarize_layers(history_out)
    with open(os.path.join(RESULTS, 'size-accounting.json'), 'w', encoding='utf-8') as fh:
        json.dump(dict(images=images_out, layers=_layer_rows), fh, ensure_ascii=False, indent=1)
    display(pd.DataFrame(_layer_rows).head(12).style.set_caption(
        'ведомость слоёв v1 (docker history); размеры — ВАШЕ измерение (size-accounting.json)'))

In [ ]:
FIXTURE_HISTORY = """45.2MB\tRUN pip install --no-cache-dir --prefix=/install -r app/requirements.txt
74.3MB\tCOPY --from=deps /install /usr/local
2.1MB\tCOPY app/ app/
0B\tWORKDIR /srv
0B\tCOPY app/requirements.txt app/requirements.txt
129.8MB\tFROM python:3.12-slim"""
# Учебная стенограмма формата `docker history`: разбор исполняется на любой машине,
# размеры здесь иллюстративные, НЕ измерение. Живое измерение — клетка выше.
_fixture_rows = summarize_layers(FIXTURE_HISTORY)
display(pd.DataFrame(_fixture_rows).style.set_caption(
    'формат docker history на учебном примере — размеры иллюстративные, не измерение'))

## Блок 3 — Экономика старта: холодный и тёплый путь

**Холодный старт** (контейнера нет — создаём заново): create + extract слоёв на диск + старт
интерпретатора + init приложения (импорты, первый ответ /health). Если слоёв нет локально —
плюс pull из registry; этот член в лабе не измеряется: он зависит от сети, а не от вашего
образа, и попадает в журнал только как объявленный член плана.

**Тёплый старт** (`docker stop` → `docker start`): слои уже распакованы — платим только
интерпретатор и init.

**Дисциплина блока:** сначала план из build-plan.json (какие члены вы ожидаете доминирующими),
потом измерение обоих путей на ВАШЕЙ машине, потом одно обоснованное изменение и повторное
измерение. Изменение без названного механизма — не улучшение, а суеверие. Оценка времени:
~2–4 мин (по два старта на образ, с прогресс-барами).

In [ ]:
def measure_startup(image: str, name: str, host_port: int, deadline_s: float = 120.0) -> dict:
    """Measure cold (create+start from local image) and warm (stop+start) time-to-healthy.

    Parameters
    ----------
    image : str
        Image tag, e.g. 't09-gw:v1'.
    name : str
        Container name; recreated on the cold path.
    host_port : int
        Host port mapped to the container port 8000.
    deadline_s : float, default 120.0
        Wall budget for each path.

    Returns
    -------
    dict
        Keys 'cold' and 'warm' mapped to waited seconds until the first 200 /health.
    """
    # Уровень B: скелет готов.
    # Шаг 1 (cold): docker('rm', '-f', name); t0; docker('run', '-d', '--name', name,
    #               '-p', f'{host_port}:8000', image)
    # Шаг 2 (cold): wait_healthy(f'http://127.0.0.1:{host_port}/health', deadline_s) → секунды
    # Шаг 3 (warm): docker('stop', name); t1; docker('start', name); снова wait_healthy
    # Шаг 4: верните dict(cold=..., warm=...) в секундах (округление до 0.1)
    raise NotImplementedError('TODO: измерьте оба пути по шагам')

if not require_docker():
    print(DOCKER_SKIP)
else:
    _startup = measure_startup('t09-gw:v1', 't09-gw', RUN_FLAGS['port_host'])
    with open(os.path.join(RESULTS, 'startup.json'), 'w', encoding='utf-8') as fh:
        json.dump(_startup, fh, ensure_ascii=False, indent=1)
    display(pd.DataFrame([dict(mode=k, wall_s=v) for k, v in _startup.items()]).style.set_caption(
        'холодный (create+extract+init) против тёплого (init) — ваш журнал (startup.json)'))

In [ ]:
# Эталонная матрица кэша для самопроверки: 1 = слой пересобирается, 0 = cache hit.
# Это прогноз, НЕ измерение: сверьте свои предсказания из build-plan.json,
# на docker-машине подтвердите живым `docker history` после правки файла.
import matplotlib
try:
    get_ipython()  # интерактивный фронтенд: инлайн-бэкенд, график виден
except NameError:
    matplotlib.use('Agg')  # headless-запуск: только файлы
import matplotlib.pyplot as plt
import numpy as np

_layer_names = ['base', 'deps', 'app code']
_edit_names = ['правка кода', 'правка requirements', 'смена тега базы']
_matrix = np.array([[0, 0, 1], [0, 1, 1], [1, 1, 1]])
fig, ax = plt.subplots(figsize=(6.4, 3.0))
ax.imshow(_matrix, cmap=matplotlib.colors.ListedColormap(['#d1e7dd', '#f8d7da']), vmin=0, vmax=1)
ax.set_xticks(range(3), _layer_names, fontsize=9)
ax.set_yticks(range(3), _edit_names, fontsize=9)
for i in range(3):
    for j in range(3):
        ax.text(j, i, 'rebuild' if _matrix[i, j] else 'cache hit',
                ha='center', va='center', fontsize=9)
ax.set_title('Эталон инвалидации кэша: пересобирается стопка ВЫШЕ правки')
fig.tight_layout()
fig.savefig(os.path.join(RESULTS, 'cache-matrix.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
IMPROVE = dict(
    change='TODO: одно изменение (у вас уже есть кандидаты: v0 → v1 — назовите его точно)',
    mechanism='TODO: названный механизм — что именно сокращает изменение',
    prediction='TODO: предсказание — какой член cold/warm сократится и почему',
)
if not require_docker():
    print(DOCKER_SKIP)
else:
    _open = [k for k, v in IMPROVE.items() if str(v).startswith('TODO')]
    if _open:
        print(f'заполните IMPROVE: {_open} — сравнение не выполнялось')
    else:
        # Оценка времени: ~2–4 мин (по холодному и тёплому старту на двух образах).
        _v0 = measure_startup('t09-gw:v0', 't09-v0', 8098)
        _v1 = measure_startup('t09-gw:v1', 't09-gw', RUN_FLAGS['port_host'])
        _rows = [dict(image='v0 naive', **_v0), dict(image='v1 lean', **_v1)]
        with open(os.path.join(RESULTS, 'improve.json'), 'w', encoding='utf-8') as fh:
            json.dump(dict(rows=_rows, improve=IMPROVE), fh, ensure_ascii=False, indent=1)
        display(pd.DataFrame(_rows).style.set_caption(
            'до/после: холодный и тёплый старт, ваше измерение (improve.json)'))

## Блок 4 — Поломки как данные: tag drift и OOM

Две намеренные поломки, каждая с вопросом «что говорит журнал?»:

1. **Tag drift.** `docker pull` несуществующего тега. Тег — не версия: плавающий `latest` меняет
байты под ногами, несуществующий — ломает сборку на ровном месте. Вопрос: какой код возврата
и какой текст ошибки — и что пиновать (тег/дайджест), чтобы этого не случалось?
2. **OOM под лимитом.** Контейнер с `--memory 64m` и процессом, который ест память кусками.
Превышение лимита — не исключение Python: ядро убивает процесс сигналом SIGKILL, и наружу видны
только exit code 137 и строки журнала. Вопрос: почему 137, а не 1?

**Дисциплина разбора:** причина → доказательство (код возврата + хвост лога) → исправление.
Ваш `diagnose_exit` (клетка ниже) формализует разбор. Ожидаемые исходы живого прогона записаны
в комментариях; на машине без Docker клетки честно пропускаются, а разбор учебных стенограмм
исполняется везде. Оценка времени живого прогона: ~30–90 с.

In [ ]:
if not require_docker():
    print(DOCKER_SKIP)
else:
    started = time.monotonic()
    # Оценка времени: ~30–90 с; python:3.12-slim может скачаться при первом прогоне.
    # Ожидаемые исходы: drill 1 → rc=1 с 'manifest unknown'; drill 2 → rc=137 (SIGKILL от cgroup).
    rc_pull, out_pull = docker('pull', 'python:3.99.9-does-not-exist')
    rc_oom, out_oom = docker('run', '--rm', '--memory', '64m', 'python:3.12-slim', 'python', '-c',
                             "data = bytearray()\nwhile True: data += b'x' * 10_000_000")
    drill = dict(pull=dict(exit_code=rc_pull, log_tail=out_pull[-200:]),
                 oom=dict(exit_code=rc_oom, log_tail=out_oom[-200:]))
    with open(os.path.join(RESULTS, 'drills.json'), 'w', encoding='utf-8') as fh:
        json.dump(drill, fh, ensure_ascii=False, indent=1)
    print(f'drills за {time.monotonic() - started:.0f} c: pull rc={rc_pull} (ожидался 1), '
          f'oom rc={rc_oom} (ожидался 137) → drills.json')

In [ ]:
def diagnose_exit(exit_code: int, log_tail: str) -> dict:
    """Classify one container failure from its exit code and log tail.

    Parameters
    ----------
    exit_code : int
        Container process exit code (e.g. 137, 1, 125, 127).
    log_tail : str
        Last log lines of the failed command (docker output tail).

    Returns
    -------
    dict
        Keys cause, evidence, fix — the S5 record for the journal.
    """
    # Уровень C: контракт — docstring выше, тела нет.
    # Ожидаемые классы: 137/SIGKILL/oom → cgroup-лимит памяти; 1 + manifest unknown/not found
    # → tag drift; 125 → ошибка самого демона; 127 → команда не найдена.
    raise NotImplementedError('TODO: реализуйте классификацию по контракту')

# Учебные стенограммы прошлых прогонов (записи, не живые данные): разбор исполняется везде.
OOM_LOG = """data grew past the 64 MiB cgroup limit
kernel: oom-kill:memcg limit exceeded, victim python killed by SIGKILL
exit status 137"""
PULL_LOG = """Error response from daemon: manifest for python:3.99.9-does-not-exist: not found
manifest unknown"""
_rows = [dict(drill='oom-64m (стенограмма)', **diagnose_exit(137, OOM_LOG)),
         dict(drill='tag-drift (стенограмма)', **diagnose_exit(1, PULL_LOG))]
_drills_path = os.path.join(RESULTS, 'drills.json')
if os.path.exists(_drills_path):  # живой прогон выше — добавляем реальные коды выхода
    with open(_drills_path, encoding='utf-8') as fh:
        _live = json.load(fh)
    _rows.append(dict(drill='oom-64m (живой)', **diagnose_exit(_live['oom']['exit_code'],
                                                              _live['oom']['log_tail'])))
    _rows.append(dict(drill='tag-drift (живой)', **diagnose_exit(_live['pull']['exit_code'],
                                                                _live['pull']['log_tail'])))
display(pd.DataFrame(_rows).style.set_caption(
    'разбор: причина → доказательство → исправление (drills.json, если живой прогон был)'))

## Stretch — SBOM и цепочка поставок

SBOM (software bill of materials) — инвентарь всего, что внутри образа: пакеты ОС, python-пакеты,
версии. Один образ — один документ. Три вопроса, которые он делает явными:

1. **Плавающий тег базы.** `python:3.12` сегодня и через полгода — разные байты при одном теге.
2. **Зависимости без границ.** requirements.txt без точных версий — лотерея на каждой сборке;
   наш requirements pinned: `fastapi==0.115.8`, `uvicorn==0.34.0`.
3. **Слои всё помнят.** Файл, удалённый следующей инструкцией, остаётся в предыдущем слое:
   `docker history` читает то, что вы «стёрли». Секрет в слое — секрет навсегда.

Инструменты: `docker sbom` (плагин) или `docker scout sbom`. Клетка ниже сохранит инвентарь
в журнал; если инструмента нет — напечатает честную инструкцию и не сделает вид, что всё
прошло. Ограничение: один инструмент и один образ — это не сертификат безопасности.

In [ ]:
SUPPLY_RISKS = dict(
    risk_1='TODO: плавающий тег базового образа — почему это риск цепочки поставок',
    risk_2='TODO: зависимости без точных версий — почему это риск',
    risk_3='TODO: что остаётся в истории слоёв после удаления файла/секрета',
)
if not require_docker():
    print(DOCKER_SKIP)
else:
    rc, out = docker('sbom', 't09-gw:v1')
    tool = 'docker sbom'
    if rc != 0:
        rc, out = docker('scout', 'sbom', 't09-gw:v1')
        tool = 'docker scout sbom'
    if rc != 0:
        print('skipped: SBOM-инструмент недоступен (docker sbom / docker scout) — '
              'риски заполните по Dockerfile и requirements.txt, инвентарь не обязателен')
    else:
        with open(os.path.join(RESULTS, 'sbom-summary.txt'), 'w', encoding='utf-8') as fh:
            fh.write(out)
        _open = sum(1 for v in SUPPLY_RISKS.values() if str(v).startswith('TODO'))
        print(f'SBOM сохранён ({tool}): sbom-summary.txt ({len(out)} B); '
              f'незаполненных рисков: {_open}/3')

## Что вы построили — и чего эта лаба не доказывает

**Артефакт:** ваш lean-образ `t09-gw:v1` (multi-stage, pinned, non-root, healthcheck) плюс журнал
в `results/local/attempt-001/`: замороженный план сборки, предсказания кэша и эталонная матрица,
probe-журнал контракта через границу, ведомость слоёв, измерения холодного/тёплого старта,
разбор двух поломок, манифест с sha256. От первого smoke-прогона до финала журнал рос вокруг
одной вещи — образа: передвижной кухни, которая едет на любой машине одинаково.

**Чему научились:** читать образ как упорядоченные слои и предсказывать инвалидацию кэша до
сборки; писать lean multi-stage Dockerfile и защищать каждый слой; проводить контракт T08 через
границу контейнера без изменений; считать экономику старта именованными членами и рубить
доминирующий; разбирать поломки по кодам возврата и логам; читать SBOM как инвентарь рисков.

**Чего НЕ доказывает эта лаба:** размеры и секунды — ваши, локальные, машинно-зависимые (курс
не цитирует их как engine-бенчмарки); мок-движок — объявленная синтетика, не vLLM; registry-
операции за пределами локальных тегов не трогались; GPU-вариант образа (CUDA base + vLLM pin)
— демонстрация владельца; Kubernetes и мульти-GPU — впереди.

**Куда дальше:** Topic 10 масштабирует движки на несколько GPU. Ничего из этого не работает,
если единица развёртывания не неизменяема: образ из этой лабы — та самая единица. Флот — это
много одинаковых кухонь; чертёж вы теперь умеете писать и защищать.

In [ ]:
artifacts = {}
for name in ('build-plan.json', 'probe-journal.json', 'startup.json', 'improve.json',
             'drills.json', 'size-accounting.json', 'sbom-summary.txt', 'cache-matrix.png'):
    path = os.path.join(RESULTS, name)
    if os.path.exists(path):
        with open(path, 'rb') as fh:
            artifacts[name] = hashlib.sha256(fh.read()).hexdigest()
with open(os.path.join(RESULTS, 'manifest.json'), 'w', encoding='utf-8') as fh:
    json.dump(dict(profile=PROFILE, attempt=ATTEMPT, artifacts=artifacts),
              fh, ensure_ascii=False, indent=1)
print(f'{len(artifacts)} артефактов хэшировано в manifest.json — журнал растущего артефакта закрыт')